In [2]:
! pip install pandas plotly sqlalchemy matplotlib rapidfuzz

Defaulting to user installation because normal site-packages is not writeable


In [4]:
! pip install scipy

Defaulting to user installation because normal site-packages is not writeable


In [3]:
! pip install fuzzywuzzy[speedup]


Defaulting to user installation because normal site-packages is not writeable


 In-demand skills or technologies------Frequency of most common job tags(job_tags) 

In [5]:
! pip install pandas plotly sqlalchemy matplotlib rapidfuzz

Defaulting to user installation because normal site-packages is not writeable


In [122]:
import pandas as pd
import plotly.express as px
from rapidfuzz import fuzz, process


In [123]:
import pandas as pd

# Load the CSV file
df = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\Work\CSV\freelancer_cleaned.csv")

# Remove duplicates based on the unique 'hash_key_value' column
df = df.drop_duplicates(subset=['hash_key_value'])

# Drop rows with missing or empty 'job_tags'
df = df.dropna(subset=['job_tags'])
df = df[df['job_tags'].str.strip() != '']

# Print the cleaned DataFrame
print(df)



        id                                          job_title  \
0        1         Consumer-focused Physical Product Designer   
1        2  Automation with artificial intelligence with w...   
2        3  AI-Powered Article Writing Assistant - 26/03/2...   
3        4                           Tech Articles on AI -- 2   
4        5    AI-Powered Algorithmic Trading Strategy Creator   
...    ...                                                ...   
4995  4996                Packaging design for snack industry   
4996  4997                    Business development consultant   
4997  4998                    Offline ChatBot engine dev -- 2   
4998  4999                                Accounting software   
4999  5000                            Power BI developer -- 2   

                                        job_description  \
0     Product Name: 4:20 Slim Unbleached Brown Rolli...   
1     Automation with artificial intelligence with w...   
2     ...✔ Maintain a Consistent Tone – Fo

In [124]:
# Clean unwanted characters
df['job_tags'] = df['job_tags'].str.replace(r'[\[\]\'\"]', '', regex=True)
print(df.job_tags)

0       Corporate Identity, Covers & Packaging, Graphi...
1       Artificial Intelligence, Customer Service Chat...
2       Article Writing, Blog, Content Writing, Copywr...
3       Article Rewriting, Article Writing, Blog, Cont...
4       JavaScript, Machine Learning (ML), Metatrader,...
                              ...                        
4995    Brochure Design, Corporate Identity, Covers & ...
4996    Business Analysis, Business Plans, Business Wr...
4997    Android, C# Programming, Java, Mobile App Deve...
4998    Accounting, Sales Account Management, Software...
4999    Data Warehousing, Database Administration, Mic...
Name: job_tags, Length: 4675, dtype: object


In [125]:
# Split and explode job tags
df['job_tags'] = df['job_tags'].str.split(',')
df_exploded = df.explode('job_tags')
print(df_exploded)

        id                                   job_title  \
0        1  Consumer-focused Physical Product Designer   
0        1  Consumer-focused Physical Product Designer   
0        1  Consumer-focused Physical Product Designer   
0        1  Consumer-focused Physical Product Designer   
0        1  Consumer-focused Physical Product Designer   
...    ...                                         ...   
4999  5000                     Power BI developer -- 2   
4999  5000                     Power BI developer -- 2   
4999  5000                     Power BI developer -- 2   
4999  5000                     Power BI developer -- 2   
4999  5000                     Power BI developer -- 2   

                                        job_description  \
0     Product Name: 4:20 Slim Unbleached Brown Rolli...   
0     Product Name: 4:20 Slim Unbleached Brown Rolli...   
0     Product Name: 4:20 Slim Unbleached Brown Rolli...   
0     Product Name: 4:20 Slim Unbleached Brown Rolli...   
0     Pr

In [126]:
# Normalize tags
df_exploded['job_tags'] = df_exploded['job_tags'].str.strip().str.lower()
df_exploded = df_exploded[df_exploded['job_tags'] != '']
print(df_exploded)

        id                                   job_title  \
0        1  Consumer-focused Physical Product Designer   
0        1  Consumer-focused Physical Product Designer   
0        1  Consumer-focused Physical Product Designer   
0        1  Consumer-focused Physical Product Designer   
0        1  Consumer-focused Physical Product Designer   
...    ...                                         ...   
4999  5000                     Power BI developer -- 2   
4999  5000                     Power BI developer -- 2   
4999  5000                     Power BI developer -- 2   
4999  5000                     Power BI developer -- 2   
4999  5000                     Power BI developer -- 2   

                                        job_description  \
0     Product Name: 4:20 Slim Unbleached Brown Rolli...   
0     Product Name: 4:20 Slim Unbleached Brown Rolli...   
0     Product Name: 4:20 Slim Unbleached Brown Rolli...   
0     Product Name: 4:20 Slim Unbleached Brown Rolli...   
0     Pr

In [127]:
# Get unique tags
unique_tags = df_exploded['job_tags'].unique().tolist()
print(unique_tags)

['corporate identity', 'covers & packaging', 'graphic design', 'logo design', 'product design', 'artificial intelligence', 'customer service chatbot', 'php', 'process automation', 'python', 'article writing', 'blog', 'content writing', 'copywriting', 'ghostwriting', 'article rewriting', 'javascript', 'machine learning (ml)', 'metatrader', 'software architecture', 'trading', 'web application', 'ai model development', 'algorithm', 'azure openai', 'microsoft azure', 'nlp', 'report writing', 'research', 'research writing', 'technical writing', 'documentation', 'website management', 'autocad architecture', 'building architecture', 'interior design', 'landscape design', 'photoshop', 'html', 'landing pages', 'user interface / ia', 'website design', 'openai', 'management', 'audio production', 'audio services', 'music', 'music management', 'sound design', '.net core', 'mvc', 'sql', 'wordpress', 'web design', 'internet marketing', 'leads', 'link building', 'marketing', 'seo', 'illustrator', 'ani

In [128]:
import pandas as pd
from fuzzywuzzy import process, fuzz
# Auto-group similar tags using fuzzy matching
canonical_map = {}
canonical_names = []

for tag in unique_tags:
    match = process.extractOne(tag, canonical_names, scorer=fuzz.token_sort_ratio)
    if match and match[1] > 85:  
        canonical_map[tag] = match[0]
    else:
        canonical_map[tag] = tag
        canonical_names.append(tag)

df_exploded['job_tags_canonical'] = df_exploded['job_tags'].map(canonical_map)
print(df_exploded[['job_tags', 'job_tags_canonical']])



                     job_tags       job_tags_canonical
0          corporate identity       corporate identity
0          covers & packaging       covers & packaging
0              graphic design           graphic design
0                 logo design              logo design
0              product design           product design
...                       ...                      ...
4999         data warehousing         data warehousing
4999  database administration  database administration
4999        microsoft powerbi        microsoft powerbi
4999                      sql                      sql
4999     statistical analysis     statistical analysis

[21303 rows x 2 columns]


In [129]:
# Replace tags with canonical names
df_exploded['normalized_tag'] = df_exploded['job_tags'].map(canonical_map)
print(df_exploded['normalized_tag'])

0            corporate identity
0            covers & packaging
0                graphic design
0                   logo design
0                product design
                 ...           
4999           data warehousing
4999    database administration
4999          microsoft powerbi
4999                        sql
4999       statistical analysis
Name: normalized_tag, Length: 21303, dtype: object


In [130]:
from fuzzywuzzy import fuzz
import pandas as pd

# Assuming 'unique_tags' is a list of your unique tags
matches = []

# Compare each pair of unique tags
for i in range(len(unique_tags)):
    for j in range(i + 1, len(unique_tags)):
        score = fuzz.token_sort_ratio(unique_tags[i], unique_tags[j])
        matches.append((unique_tags[i], unique_tags[j], score))

# Convert matches to a DataFrame to inspect
matches_df = pd.DataFrame(matches, columns=['Tag1', 'Tag2', 'Score'])

# Print the DataFrame to see tag comparisons
print(matches_df)

# Count frequency of tags (as in your original code)
tag_counts = df_exploded['normalized_tag'].value_counts().reset_index()
tag_counts.columns = ['job_tag', 'count']

# Print all tag frequencies
print("\nAll Job Tags Frequency :\n")
print(tag_counts.to_string(index=False))


                       Tag1                      Tag2  Score
0        corporate identity        covers & packaging     41
1        corporate identity            graphic design     25
2        corporate identity               logo design     21
3        corporate identity            product design     25
4        corporate identity   artificial intelligence     34
...                     ...                       ...    ...
678025        java tutoring       telegram moderation     31
678026        java tutoring  sales account management     27
678027             makerbot       telegram moderation     37
678028             makerbot  sales account management     25
678029  telegram moderation  sales account management     33

[678030 rows x 3 columns]

All Job Tags Frequency :

                                                    job_tag  count
                                             graphic design    945
                                                illustrator    759
             

In [131]:
# Plotting
fig = px.pie(
    names=tag_counts['job_tag'].head(15),
    values=tag_counts['count'].head(15),
    title='Top 15 Most Demanded Job Tags ',
    width=900,
    height=700
)

fig.update_traces(textinfo='percent+label', textposition='inside')
fig.show()


which roles command higher bids-----Average Bid Amount by Job Title. (bit_amount and job_title)

In [132]:
import pandas as pd
import plotly.express as px
import numpy as np

In [133]:
# Load the data 
df=pd.read_csv('C:/Users/USER/OneDrive/Desktop/freelancer_cleaned_output.csv')
print(df)

                             id                  job_title  \
0        ""Covers & Packaging""         ""Graphic Design""   
1                       41 bids                   Verified   
2               ""Copywriting""   ""Norwegian Translator""   
3                ""Blockchain""       ""Research Writing""   
4              ""Illustration""            ""Illustrator""   
..                          ...                        ...   
135   ""Machine Learning (ML)""                 ""Python""   
136         ""Content Writing""            ""Copywriting""   
137                   ""MySQL""                    ""PHP""   
138                    ""HTML""                  ""MySQL""   
139      ""Corporate Identity""     ""Covers & Packaging""   

                job_description                  job_tags  \
0               ""Logo Design""      ""Product Design""]"   
1               3/27/2025 10:54   artificial intelligence   
2              ""Proofreading""         ""Translation""]"   
3      ""So

In [134]:
import pandas as pd
import plotly.express as px
import re
import numpy as np

# Load the dataset
df = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\Work\CSV\freelancer_cleaned.csv")

# Remove duplicates based on the unique 'hash_key_value' column
df = df.drop_duplicates(subset=['hash_key_value'])

# Drop rows with missing or empty 'job_title' or 'bit_amount'
df = df.dropna(subset=['job_title', 'bit_amount'])
df = df[df['job_title'].str.strip() != '']
df = df[df['bit_amount'].astype(str).str.strip() != '']

# Convert bit_amount to string
df['bit_amount'] = df['bit_amount'].astype(str)

# Remove hourly entries
df = df[~df['bit_amount'].str.contains('/hour', case=False, na=False)]

# Clean bit_amount values
def clean_bit_amount(val):
    # Remove 'avg bid', currency symbols, commas, letters, etc.
    val = re.sub(r'avg bid', '', val, flags=re.IGNORECASE)
    val = re.sub(r'[^\d\.\-\s/]', '', val)  

    val = val.strip()

    # Handle ranges like "20 - 50"
    if '-' in val:
        parts = val.split('-')
        try:
            nums = [float(p.strip()) for p in parts if p.strip().replace('.', '', 1).isdigit()]
            if len(nums) == 2:
                return np.mean(nums)
        except:
            return np.nan

    # Single float value
    try:
        return float(val)
    except:
        return np.nan

# Apply cleaning
df['bit_amount_clean'] = df['bit_amount'].apply(clean_bit_amount)

# Drop rows where bit_amount_clean is NaN
df = df.dropna(subset=['bit_amount_clean'])

# Clean and simplify job_title
def clean_title(title):
    title = re.sub(r'avg bid', '', title, flags=re.IGNORECASE)  
    title = re.sub(r'[^a-zA-Z\s]', '', title) 
    title = re.sub(r'\d+', '', title)  
    title = re.sub(r'\s+', ' ', title).strip().lower() 
    return ' '.join(title.split()[:2]) 

df['job_title_clean'] = df['job_title'].apply(clean_title)

# Compute average bid per job title 
avg_bids = df.groupby('job_title_clean')['bit_amount_clean'].mean().reset_index()
avg_bids = avg_bids.sort_values(by='bit_amount_clean', ascending=False)

# Show top N job titles
top_n = 15
avg_bids_top = avg_bids.head(top_n)

# Bar chart
fig = px.bar(
    avg_bids_top,
    x='bit_amount_clean',         
    y='job_title_clean',          
    color='job_title_clean',
    orientation='h',              
    title=f'Top {top_n} Job Titles by Average Bid Amount',
    labels={'job_title_clean': 'Job Title', 'bit_amount_clean': 'Average Bid ($)'},
    text='bit_amount_clean',
    color_discrete_sequence=px.colors.qualitative.Set1,
    height=900,
    width=1300
)
fig.show()


competition for jobs------Number of Bids per Job.(number_of_bits and job_title)

In [135]:
import pandas as pd
import plotly.express as px
import re
import numpy as np

In [136]:
# Load the dataset
df = pd.read_csv('C:/Users/USER/OneDrive/Desktop/freelancer_cleaned_output.csv')

In [137]:
# Clean 'number_of_bits' column
def clean_num_bids(val):
    val = str(val)
    val = re.sub(r'[^\d]', '', val)  
    return int(val) if val.isdigit() else None
df['number_of_bits_clean'] = df['number_of_bits'].apply(clean_num_bids)
print(df['number_of_bits_clean'])

0        4.0
1        NaN
2       11.0
3       18.0
4      701.0
       ...  
135     11.0
136      9.0
137     28.0
138     19.0
139     42.0
Name: number_of_bits_clean, Length: 140, dtype: float64


In [138]:

# Drop rows where number_of_bits is missing or not numeric
df = df.dropna(subset=['number_of_bits_clean'])
print(df)

                                     id                  job_title  \
0                ""Covers & Packaging""         ""Graphic Design""   
2                       ""Copywriting""   ""Norwegian Translator""   
3                        ""Blockchain""       ""Research Writing""   
4                      ""Illustration""            ""Illustrator""   
5     ""Counselling and Psychotherapy""           ""Ghostwriting""   
..                                  ...                        ...   
135           ""Machine Learning (ML)""                 ""Python""   
136                 ""Content Writing""            ""Copywriting""   
137                           ""MySQL""                    ""PHP""   
138                            ""HTML""                  ""MySQL""   
139              ""Corporate Identity""     ""Covers & Packaging""   

                job_description                  job_tags days_left_to_expire  \
0               ""Logo Design""      ""Product Design""]"         6 days left 

In [139]:
# Clean and simplify job_title 
def clean_title(title):
    title = re.sub(r'avg bid', '', str(title), flags=re.IGNORECASE)
    title = re.sub(r'[^a-zA-Z\s]', '', title)  
    title = re.sub(r'\d+', '', title)          
    title = re.sub(r'\s+', ' ', title).strip().lower() 
    return title
df['job_title_clean'] = df['job_title'].apply(clean_title)
print(df['job_title_clean'])

0            graphic design
2      norwegian translator
3          research writing
4               illustrator
5              ghostwriting
               ...         
135                  python
136             copywriting
137                     php
138                   mysql
139        covers packaging
Name: job_title_clean, Length: 139, dtype: object


In [140]:
# Group by cleaned job title and compute average number of bids 
avg_bids_per_title = df.groupby('job_title_clean')['number_of_bits_clean'].mean().reset_index()
avg_bids_per_title = avg_bids_per_title.sort_values(by='number_of_bits_clean', ascending=False)
print(avg_bids_per_title)

            job_title_clean  number_of_bits_clean
36              illustrator            357.384615
31           graphic design            164.500000
35             illustration            161.000000
3                 animation            152.000000
66        user interface ia            112.000000
..                      ...                   ...
16                      crm              7.000000
19  database administration              4.000000
0       aircraft propulsion              3.000000
34          human resources              2.000000
68         video production              0.000000

[71 rows x 2 columns]


In [141]:
# Top N titles 
top_n = 15
avg_bids_per_title_top = avg_bids_per_title.head(top_n)
# Plot
fig = px.bar(
    avg_bids_per_title_top,
    x='job_title_clean',
    y='number_of_bits_clean',
    color='job_title_clean',
    title=f'Top {top_n} Job Titles by Average Number of Bids',
    labels={'job_title_clean': 'Job Title', 'number_of_bits_clean': 'Average Number of Bids'},
    text='number_of_bits_clean',
    color_discrete_sequence=px.colors.qualitative.Set2,
    width=1300,
    height=700
)

fig.update_traces(texttemplate='%{text:.0f}', textposition='outside')
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()